## 1. Setup

In [1]:
%load_ext autoreload
%autoreload 2

import os
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import torch
import torch.nn as nn
from dotenv import load_dotenv
from torch.utils.data import DataLoader

load_dotenv()
DATA_ROOT = Path(os.getenv("DATA_ROOT"))
NIH_DIR = DATA_ROOT / "NIH_Chest_X-Rays"
PROJECT_ROOT = Path.cwd().parent
DEVICE = "cuda"

print(torch.cuda.get_device_name(0))

torch.manual_seed(0)

NVIDIA GeForce RTX 5090 Laptop GPU


## 2. Build and Inspect Model

In [2]:
from src.model import build_model

model = build_model().to(DEVICE).to(memory_format=torch.channels_last)

n_total = sum(p.numel() for p in model.parameters())
n_head = sum(p.numel() for p in model.classifier.parameters())
print(model.classifier)
print(f"Total params: {n_total:,}  |  Head params: {n_head:,}  ({n_head / n_total:.2%})")

Linear(in_features=1024, out_features=14, bias=True)
Total params: 6,968,206  |  Head params: 14,350  (0.21%)


## 3. Grab One Fixed Batch

In [3]:
from src.data import NIH_DISEASES, build_image_paths, load_dev_labels
from src.dataset import NIHDataset
from src.paths import SPLIT_CSV
from src.transforms import build_transforms

df = load_dev_labels(NIH_DIR, SPLIT_CSV)
train_ds = NIHDataset(
    df[df["split"] == "train"], build_image_paths(NIH_DIR), build_transforms(train=False)
)

torch.manual_seed(0)
x, y = next(iter(DataLoader(train_ds, batch_size=32, shuffle=True)))
x = x.to(DEVICE).to(memory_format=torch.channels_last)
y = y.to(DEVICE)

print(x.shape, y.shape, f"positives in batch: {int(y.sum())}")

torch.Size([32, 3, 224, 224]) torch.Size([32, 14]) positives in batch: 23


## 4. Sanity Check 1, Init Loss

In [4]:
loss_fn = nn.BCEWithLogitsLoss()

model.eval()
with torch.no_grad():
    logits = model(x)

print("logits:", logits.shape, logits.dtype, f"mean={logits.mean():.3f} std={logits.std():.3f}")
print(f"initial loss = {loss_fn(logits, y):.4f}   (ln 2 = {torch.log(torch.tensor(2.0)):.4f})")
print("example probabilities (image 0):", [f"{p:.2f}" for p in torch.sigmoid(logits[0]).tolist()])

logits: torch.Size([32, 14]) torch.float32 mean=0.147 std=0.433
initial loss = 0.7882   (ln 2 = 0.6931)
example probabilities (image 0): ['0.44', '0.52', '0.59', '0.47', '0.69', '0.58', '0.56', '0.61', '0.46', '0.50', '0.39', '0.42', '0.63', '0.67']


## 5. Sanity Check 2, Overfit One Batch

In [5]:
model.train()
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

n_steps = 150
start = time.perf_counter()
for step in range(1, n_steps + 1):
    with torch.autocast("cuda", dtype=torch.bfloat16):
        logits = model(x)
    loss = loss_fn(logits.float(), y)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if step % 25 == 0:
        print(f"step {step:3d}  loss {loss.item():.5f}")
torch.cuda.synchronize()
elapsed = time.perf_counter() - start

model.eval()
with torch.no_grad():
    preds = (torch.sigmoid(model(x)) > 0.5).float()
print(f"\nbatch label accuracy after overfitting: {(preds == y).float().mean():.3f}")
print(f"GPU training throughput: ~{n_steps * len(x) / elapsed:.0f} img/s (batch 32, bf16)")

step  25  loss 0.04201
step  50  loss 0.01325
step  75  loss 0.00797
step 100  loss 0.00568
step 125  loss 0.00433
step 150  loss 0.00343

batch label accuracy after overfitting: 1.000
GPU training throughput: ~604 img/s (batch 32, bf16)


## Model decisions
- Backbone: DenseNet-121, ImageNet-pretrained (`IMAGENET1K_V1`). Standard NIH ChestX-ray14 baseline (CheXNet) for literature comparability; ~7M params allows fast iteration.
- Full fine-tuning (no frozen layers): X-rays are far from the natural-image domain.
- Head: `Linear(1024 → 14)`, one logit per disease, no sigmoid in the model; `BCEWithLogitsLoss` applies the sigmoid stably.
- Global average pooling + linear head enables CAM/Grad-CAM in milestone 8.
- Sanity checks: initial loss ≈ ln 2 (0.66 vs 0.693; varies slightly with random head init) ✓; single-batch overfit 0.66 → ~0.003 ✓.
- bf16 autocast + channels_last: ~900 img/s training at batch 64 (separate benchmark; ~565 img/s at batch 32 here) vs ~106 img/s fp32; roughly matches DataLoader throughput (~700–900 img/s), so no image cache needed yet.